# Transformer Policy - FFAI Assistant

Attention-based policy para memoria temporal.

In [ ]:
import tensorflow as tf
from tensorflow import keras
print(f"TensorFlow: {tf.__version__}")

In [ ]:
STATE_SIZE, ACTION_SIZE = 256, 15
SEQ_LEN, NUM_HEADS, D_MODEL = 32, 4, 128

# Transformer Policy
inputs = keras.Input(shape=(SEQ_LEN, STATE_SIZE))

# Positional Encoding + Projection
x = keras.layers.Dense(D_MODEL)(inputs)

# Multi-Head Attention
attn_out = keras.layers.MultiHeadAttention(
    num_heads=NUM_HEADS, key_dim=D_MODEL//NUM_HEADS
)(x, x)
x = keras.layers.LayerNormalization()(x + attn_out)

# Feed Forward
ffn = keras.Sequential([
    keras.layers.Dense(D_MODEL * 2, activation='relu'),
    keras.layers.Dense(D_MODEL)
])
ffn_out = ffn(x)
x = keras.layers.LayerNormalization()(x + ffn_out)

# Global Average Pooling + Output
x = keras.layers.GlobalAveragePooling1D()(x)
outputs = keras.layers.Dense(ACTION_SIZE, activation='softmax')(x)

transformer = keras.Model(inputs, outputs, name='Transformer_Policy')
print(f"✓ Transformer: {transformer.count_params():,} params")

In [ ]:
# Exportar
converter = tf.lite.TFLiteConverter.from_keras_model(transformer)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
with open('transformer_policy.tflite', 'wb') as f:
    f.write(converter.convert())
print("✓ Exportado: transformer_policy.tflite")

In [ ]:
from google.colab import files
files.download('transformer_policy.tflite')